# Panorama Stitching Project
### Lorenzo Brizzi

1) LAB- "Matching": apply the results of your matching procedure to image registration or the construction of a mosaic / panoramic image
you are required to use the correspondences to estimate a global transformation (see https://docs.opencv.org/4.x/d9/dab/tutorial_homography.html) you will use to register one image with respect to another

In [1]:
import cv2
import numpy as np

def create_panorama(img1, img2):
    """
    Creates a simple left-right panorama (mosaic) of img1 and img2
    by finding a homography that warps img2 into img1's coordinate system.
    It saves:
      1) img1 with keypoints
      2) img2 with keypoints
      3) an image showing all (good) matches
      4) the final panorama
    """
    # 1. Convert to grayscale
    gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

    # 2. Detect keypoints and descriptors
    sift = cv2.SIFT_create() 
    kp1, des1 = sift.detectAndCompute(gray1, None)
    kp2, des2 = sift.detectAndCompute(gray2, None)

    # -- Draw and save images with keypoints
    img1_with_kp = cv2.drawKeypoints(
        img1, kp1, None, color=(0, 255, 0),
        flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
    )
    img2_with_kp = cv2.drawKeypoints(
        img2, kp2, None, color=(0, 255, 0),
        flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
    )
    cv2.imwrite("debug_img1_keypoints.jpg", img1_with_kp)
    cv2.imwrite("debug_img2_keypoints.jpg", img2_with_kp)

    # 3. Match descriptors with BFMatcher (KNN)
    bf = cv2.BFMatcher()
    matches = bf.knnMatch(des1, des2, k=2)

    # 4. Filter matches using Lowe's ratio test
    good = []
    ratio_thresh = 0.75
    for m, n in matches:
        if m.distance < ratio_thresh * n.distance:
            good.append(m)
    
    MIN_MATCH_COUNT = 8
    if len(good) < MIN_MATCH_COUNT:
        print(f"Not enough good matches: {len(good)}/{MIN_MATCH_COUNT}")
        return None

    # -- Draw and save an image of the good matches
    match_img = cv2.drawMatches(
        img1, kp1, img2, kp2, good, None,
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
    )
    cv2.imwrite("debug_good_matches.jpg", match_img)

    # 5. Estimate Homography (warp img2 -> img1)
    src_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    H, _ = cv2.findHomography(dst_pts, src_pts, cv2.RANSAC, 5.0)

    # 6. Warp img2 into img1's coordinate space
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]

    corners_img2 = np.float32([[0,0],[w2,0],[w2,h2],[0,h2]]).reshape(-1,1,2)
    warped_corners_img2 = cv2.perspectiveTransform(corners_img2, H)

    corners_img1 = np.float32([[0,0],[w1,0],[w1,h1],[0,h1]]).reshape(-1,1,2)
    all_corners = np.concatenate((corners_img1, warped_corners_img2), axis=0)

    [xmin, ymin] = np.int32(all_corners.min(axis=0).ravel() - 0.5)
    [xmax, ymax] = np.int32(all_corners.max(axis=0).ravel() + 0.5)

    # Shift any negative coords
    t = [-xmin, -ymin]
    H_translate = np.array([[1, 0, t[0]],
                            [0, 1, t[1]],
                            [0, 0, 1]], dtype=np.float32)

    panorama_width = xmax - xmin
    panorama_height = ymax - ymin

    warped_img2 = cv2.warpPerspective(
        img2, H_translate @ H,
        (panorama_width, panorama_height)
    )

    # 7. Compose the final panorama
    panorama = np.zeros((panorama_height, panorama_width, 3), dtype=np.uint8)
    panorama[t[1]:h1 + t[1], t[0]:w1 + t[0]] = img1

    mask = (warped_img2 > 0)
    panorama[mask] = warped_img2[mask]

    cv2.imwrite("panorama_result.jpg", panorama)
    return panorama


if __name__ == "__main__":
    img1 = cv2.imread("images/img487.jpg")
    img2 = cv2.imread("images/img488.jpg")

    if img1 is None or img2 is None:
        print("Error loading images. Check paths.")
        exit()

    pano = create_panorama(img1, img2)
    if pano is not None:
        cv2.imshow("Panorama", pano)
        cv2.waitKey(0)
        cv2.destroyAllWindows()
    else:
        print("Panorama creation failed.")


## 1. **Goal of the Lab**

The goal is to take two overlapping images (call them **`img1`** and **`img2`**) of the same scene and create a **single panoramic image** (a left‐right mosaic). 

The main tasks are:

1. **Detect and describe** distinctive points (features) in both images.  
2. **Match** those features across the two images.  
3. **Use a homography** to “warp” one image so that it aligns with the other.  
4. **Combine** (“stitch”) the warped image and the reference image into one panorama.

---

## 2. **Key Steps in the Code**

### 2.1 Converting to Grayscale

```python
gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)
```
Color information, as in almost all feature detectors in OpenCV, is unnecessary for standard keypoint detection 

### 2.2 Feature Detection & Description (SIFT)

```python
sift = cv2.SIFT_create()
kp1, des1 = sift.detectAndCompute(gray1, None)
kp2, des2 = sift.detectAndCompute(gray2, None)
```
I used **SIFT** (Scale‐Invariant Feature Transform) to:
  1. **Detect keypoints** (distinctive points / patches in the image).  
  2. **Compute descriptors** (128‐dimensional vectors for each keypoint).  

SIFT could be a good choice cause is robust to scale, rotation, and moderate changes in viewpoint/illumination. An alternative could have been ORB, 
whose descriptors are more compact and faster to compute, even if slightly less invariant to strong scale or illumination changes.

### 2.3 Drawing & Saving Keypoints (Lab Enhancement)

I drew circles around the detected keypoints in each image and save them. This visualizes where SIFT found distinctive features. 

```python
img1_with_kp = cv2.drawKeypoints(img1, kp1, ...)
img2_with_kp = cv2.drawKeypoints(img2, kp2, ...)
cv2.imwrite("debug_img1_keypoints.jpg", img1_with_kp)
cv2.imwrite("debug_img2_keypoints.jpg", img2_with_kp)
```

### 2.4 Matching Descriptors (BFMatcher + Ratio Test)

```python
bf = cv2.BFMatcher()
matches = bf.knnMatch(des1, des2, k=2)
```

I used a **brute‐force matcher** (`BFMatcher`) in KNN mode (`k=2`), which finds the two best matches for each keypoint descriptor in `img1` vs. the descriptor set in `img2`.

#### Lowe’s Ratio Test

```python
good = []
for m, n in matches:
    if m.distance < ratio_thresh * n.distance:
        good.append(m)
```

**Lowe’s ratio test** helps remove ambiguous or poor matches.  If the closest match is much better than the second‐closest, we keep it. Otherwise, it’s likely not a reliable match.  

### 2.5 Drawing & Saving the “Good Matches”

```python
match_img = cv2.drawMatches(img1, kp1, img2, kp2, good, None, ...)
cv2.imwrite("debug_good_matches.jpg", match_img)
```

This produces an image that shows lines connecting each matched keypoint in `img1` to its corresponding keypoint in `img2`.  That's useful for debugging and ensuring that I have enough correct matches.

### 2.6 Computing the Homography (RANSAC)

```python
src_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1,1,2)
dst_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1,1,2)

H, _ = cv2.findHomography(dst_pts, src_pts, cv2.RANSAC, 5.0)
```

I now have pairs of 2D point correspondences: 
  - **`dst_pts`** are coordinates in `img2`.  
  - **`src_pts`** are coordinates in `img1`.

Since I want to **warp `img2` → `img1`**, I find a homography \( H \) such that: 


$$
\begin{bmatrix}
x_{\text{img1}} \\
y_{\text{img1}} \\
1
\end{bmatrix}
\quad = \quad
H
\begin{bmatrix}
x_{\text{img2}} \\
y_{\text{img2}} \\
1
\end{bmatrix}
$$



To discard outliers, I used RANSAC, a robust method that repeatedly samples subsets of matching points to hypothesize a model (in this case, a homography) and rejects points that do not fit within a specified error threshold. By focusing on the largest consensus set, RANSAC effectively mitigates the impact of mismatches, preventing them from distorting the final alignment.

### 2.7 Warping the Second Image

```python
warped_img2 = cv2.warpPerspective(img2, H_translate @ H, (panorama_width, panorama_height))
```
Once I have \( H \), I can **warp** `img2` into `img1`’s coordinate space. This step re‐projects `img2` so that it aligns with the geometry of `img1`.

### 2.8 Bounding Box Calculation & Translation

I figured out the new bounding box (the “canvas” size) by transforming the **corner points** of `img2` and combining with the corners of `img1`.  
If `img2` extends to the left of (0,0), or if the warp causes negative coordinates, I apply a translation matrix so that everything fits in a **positive** coordinate system.  
The final canvas width/height are derived from the min and max x/y among all corners.  

### 2.9 Merging into a Single Panorama

```python
panorama = np.zeros((panorama_height, panorama_width, 3), dtype=np.uint8)
panorama[t[1]:h1 + t[1], t[0]:w1 + t[0]] = img1

mask = (warped_img2 > 0)
panorama[mask] = warped_img2[mask]
```
  
I create an **empty** (black) image with the bounding box dimensions. Then I copy `img1` to the canvas with the correct offset. Finally, I overlay the warped pixels of `img2`. If `warped_img2` is not black at a coordinate, I set the pixel of the panorama to that value.

### 2.10 Saving the Result

Finally, I save the panorama to disk:

```python
cv2.imwrite("panorama_result.jpg", panorama)
```
---